# Step 5~6 | 대시보드 완성 & HTML 저장
> **Team Sparta | 주식/코인 모니터링 프로젝트**

---

## 이번 노트북에서 만들 것
1. 앞에서 만든 함수들을 하나의 `dashboard.py` 로 합치기
2. plotly로 여러 차트를 한 화면에 배치하기
3. 완성된 대시보드를 **HTML 파일**로 저장해서 브라우저에서 열기
4. 최종 결과물 확인

>  **이번 노트북이 끝나면 `stock_dashboard.html` 파일 하나로 누구에게나 공유할 수 있습니다.**  
> 브라우저만 있으면 열리고, 마우스로 차트를 조작할 수 있어요.

---

### 최종 결과물 미리보기
```
stock_dashboard.html 을 브라우저에서 열면:

┌─────────────────────────────────────────────────────┐
│  주식 모니터링 대시보드          생성: 2024-03-20  │
├───────────────┬───────────────┬─────────────────────┤
│  현재가        │  등락가        │  등락률              │
│  $185.20      │  +5.20        │  +2.89%             │
├───────────────┴───────────────┴─────────────────────┤
│  [종가 라인 차트 + MA5/MA20 이동평균선]               │
├─────────────────────────────────────────────────────┤
│  [거래량 바차트 — 상승 빨강 / 하락 파랑]               │
├─────────────────────────────────────────────────────┤
│  [종목 비교 차트 — AAPL vs TSLA vs GOOGL]            │
└─────────────────────────────────────────────────────┘
```

---
## 이전 노트북 함수 재사용

01, 02번 노트북에서 만든 함수를 그대로 가져옵니다.  
아래 셀을 **수정하지 말고 그대로 실행**하세요.

In [ ]:
#schedule 라이브러리 install
import sys
!{sys.executable} -m pip install schedule


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ── 데이터 함수 (01번 노트북에서 가져옴) ──────────────────────

def get_stock_info(ticker_symbol):
    """종목 현재 정보 조회"""
    try:
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info
        result = {
            'name':          info.get('shortName', ticker_symbol),
            'current_price': info.get('currentPrice', 0),
            'prev_close':    info.get('previousClose', 0),
            'currency':      info.get('currency', 'USD'),
        }
        
        
        return result if result['current_price'] != 0 else None
    except Exception as e:
        print(f'[오류] {ticker_symbol}: {e}')
        return None

def get_stock_history(ticker_symbol, period='1달'):
    """기간별 가격 히스토리 조회"""
    period_map = {'1주': '1wk', '1달': '1mo', '3달': '3mo', '1년': '1y'}
    try:
        ticker = yf.Ticker(ticker_symbol)
        df = ticker.history(period=period_map.get(period, '1mo'))
        return df[['Open', 'High', 'Low', 'Close', 'Volume']] if not df.empty else None
    except Exception as e:
        print(f'[오류] {ticker_symbol}: {e}')
        return None

def calculate_change(current, prev):
    """등락가, 등락률 계산"""
    if prev == 0: return 0, 0
    change = round(current - prev, 2)
    change_pct = round((change / prev) * 100, 2)
    return change, change_pct

def calculate_moving_average(df, periods=[5, 20]):
    """이동평균 계산"""
    result = df.copy()
    for p in periods:
        result[f'MA{p}'] = result['Close'].rolling(window=p).mean()
    return result

# ── 차트 함수 (02번 노트북에서 가져옴) ──────────────────────────

def make_combined_chart(df, ticker_symbol, period, show_ma=True):
    """가격 + 거래량 통합 차트"""
    if show_ma:
        df = calculate_moving_average(df, periods=[5, 20])
    fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3],
                        shared_xaxes=True, vertical_spacing=0.04)
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines',
                              name='종가', line=dict(color='#2196F3', width=2)), row=1, col=1)
   
    if show_ma:
        for col, color in [('MA5', '#FF9800'), ('MA20', '#9C27B0')]:
            fig.add_trace(go.Scatter(x=df.index, y=df[col], mode='lines',
                                     name=col, line=dict(color=color, width=1.5, dash='dot')), row=1, col=1)
    colors = ['#E53935' if c >= o else '#1E88E5'
              for c, o in zip(df['Close'], df['Open'])]
    fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='거래량',
                         marker_color=colors, showlegend=False), row=2, col=1)
    fig.update_layout(title=f'{ticker_symbol} 주가 차트 ({period})',
                      height=480, hovermode='x unified',
                      plot_bgcolor='white', paper_bgcolor='white')
    fig.update_yaxes(title_text='가격', row=1, col=1)
    fig.update_yaxes(title_text='거래량', row=2, col=1)
    return fig

def make_comparison_chart(tickers, period='3달'):
    """여러 종목 정규화 비교 차트"""
    fig = go.Figure()
    for sym in tickers:
        df = get_stock_history(sym, period)
        if df is None: continue
        normalized = df['Close'] / df['Close'].iloc[0] * 100
        fig.add_trace(go.Scatter(x=df.index, y=normalized,
                                  mode='lines', name=sym))
    fig.add_hline(y=100, line_dash='dash', line_color='gray',
                  annotation_text='기준(100)')
    fig.update_layout(title=f'종목 비교 ({period}, 시작가=100 기준)',
                      yaxis_title='정규화 가격', height=380,
                      hovermode='x unified',
                      plot_bgcolor='white', paper_bgcolor='white')
    return fig


#2. 캔들스틱 차트(`go.Candlestick`)로 교체
def make_candle_chart(thickers , hist):
    fig = go.Figure()
    df = hist
    print('make_candle_chart hist --> ',hist)
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        increasing=dict(line=dict(color='red'), fillcolor='red'),
        decreasing=dict(line=dict(color='blue'), fillcolor='blue'),
        name=thickers
    ))
    
    fig.update_layout(title=f'캔들차트 ',
                      yaxis_title='가격', height=500,
                      hovermode='x unified',
                      plot_bgcolor='white', paper_bgcolor='white')
    
    return fig



#schedule 라이브러리 사용하여 매일 9시에 생성




print('모든 함수 로드 완료')

모든 함수 로드 완료


---
## 대시보드 설정

아래 셀에서 **조회할 종목과 기간**을 직접 바꿔보세요.

In [9]:
# 여기서 설정값을 바꿔보세요!

# 메인 종목 (차트 + 메트릭 카드에 표시)
MAIN_TICKER = '005930.KS'          # 힌트: 'AAPL' 또는 '005930.KS' (삼성전자)

# 조회 기간
PERIOD = '3달'               # 힌트: '1주', '1달', '3달', '1년' 중 하나

# 비교 차트에 표시할 종목 리스트
COMPARE_TICKERS = ['AAPL', 'GOOGL', '000660.KS']   # 힌트: ['AAPL', 'TSLA', 'GOOGL']. 000660.KS -> sk하이닉스

# 이동평균선 표시 여부
SHOW_MA = True                # 힌트: True 또는 False

# 저장할 HTML 파일 이름
OUTPUT_FILE = 'stock_dashboard.html'

print(f'설정 완료: {MAIN_TICKER} / {PERIOD} / 비교종목={COMPARE_TICKERS}')

설정 완료: 005930.KS / 3달 / 비교종목=['AAPL', 'GOOGL', '000660.KS']


---
## 데이터 수집

설정한 종목의 데이터를 한꺼번에 받아옵니다.

In [11]:
print(f'{MAIN_TICKER} 데이터 수집 중...')

# 메인 종목 정보 조회
# 힌트: get_stock_info(MAIN_TICKER)
info = get_stock_info(MAIN_TICKER)
print('info ==> ',info)

# 메인 종목 히스토리 조회
# 힌트: get_stock_history(MAIN_TICKER, PERIOD)
hist = get_stock_history(MAIN_TICKER, PERIOD)

# 데이터 확인
if info is None or hist is None:
    print(' 데이터를 가져올 수 없습니다. MAIN_TICKER 값을 확인하세요.')
else:
    print(f' {info["name"]} — 현재가: {info["current_price"]:,.2f} {info["currency"]}')
    print(f'   히스토리: {len(hist)}일치 데이터 수집 완료')

005930.KS 데이터 수집 중...
info ==>  {'name': 'SamsungElec', 'current_price': 270500.0, 'prev_close': 296000.0, 'currency': 'KRW'}
 SamsungElec — 현재가: 270,500.00 KRW
   히스토리: 59일치 데이터 수집 완료


---
## 함수 8 | `make_metric_html()` — 메트릭 카드 HTML 생성

plotly에는 '카드' 컴포넌트가 없습니다.  
대신 HTML 문자열을 직접 만들어서 대시보드에 삽입합니다.

### 요구사항
- 현재가 / 등락가 / 등락률 3개 카드
- 상승이면 빨강(`#E53935`), 하락이면 파랑(`#1E88E5`)
- 반환: HTML 문자열

In [74]:
def make_metric_html(info):
    """
    현재가 / 등락가 / 등락률 메트릭 카드 HTML을 생성합니다.

    Args:
        info (dict): get_stock_info()의 반환값

    Returns:
        str: 메트릭 카드 HTML 문자열
    """
    change, change_pct = calculate_change(
        info['current_price'], info['prev_close']
    )

    # 상승/하락에 따라 색상 결정
    # 힌트: change >= 0 이면 빨강('#E53935'), 아니면 파랑('#1E88E5')
    color = '#E53935' if change >= 0 else '#1E88E5'

    # 등락가 앞에 +/- 기호 붙이기
    # 힌트: f'{change:+.2f}' → +5.20 또는 -3.10 형태
    change_str  = f'{change:+.2f}'
    pct_str     = f'{change_pct:+.2f}%'
    
    print('change_str 등락가 => ',change_str)
    print('pct_str 등락률 => ',pct_str)
    
    


    # 메트릭 카드 하나를 만드는 내부 함수
    def card(label, value, value_color='#1a1a2e'):
        return f'''
        <div style="
            background: white;
            border: 1px solid #e0e0e0;
            border-radius: 10px;
            padding: 20px 28px;
            min-width: 180px;
            text-align: center;
        ">
            <div style="font-size:13px; color:#888; margin-bottom:8px;">{label}</div>
            <div style="font-size:26px; font-weight:700; color:{value_color};">{value}</div>
        </div>'''

    # 카드 3개를 가로로 배치
    html = f'''
    <div style="
        display: flex;
        gap: 16px;
        justify-content: center;
        flex-wrap: wrap;
        padding: 20px 0;
        background: #f8f9fa;
        border-radius: 12px;
        margin-bottom: 10px;
    ">
        {card(f'현재가 ({info["currency"]})',
              f'{info["current_price"]:,.2f}')}
        {card('전일 대비 등락가',
              change_str,
              value_color=color)}
        {card('등락률',
              pct_str,
              value_color=color)}
    </div>
    '''
    # 힌트: 두 곳 모두 color 변수 사용
    return html

In [23]:
# ✅ 테스트 — 노트북 안에서 HTML 렌더링 확인
from IPython.display import HTML

metric_html = make_metric_html(info)
HTML(metric_html)   # 노트북 안에 카드가 렌더링되면 성공!

change_str 등락가 =>  -25500.00
pct_str 등락률 =>  -8.61%


---
## 함수 9 | `build_dashboard()` — 전체 대시보드 조립

메트릭 카드 HTML + 통합 차트 + 비교 차트를 하나의 HTML 파일로 합칩니다.

### 요구사항
- 입력: 메인 종목, 기간, 비교 종목 리스트, 이동평균 여부
- 출력: HTML 문자열 (파일 저장용)
- 구성: 제목 → 메트릭 카드 → 통합 차트 → 비교 차트 순서

In [75]:
def build_dashboard(main_ticker, period, compare_tickers, show_ma=True):
    """
    전체 대시보드 HTML을 조립합니다.

    Args:
        main_ticker (str)      : 메인 종목 티커
        period (str)           : 조회 기간
        compare_tickers (list) : 비교 종목 리스트
        show_ma (bool)         : 이동평균선 표시 여부

    Returns:
        str: 완성된 대시보드 HTML 문자열
    """
    print('1/4 데이터 수집 중...')
    print('build_dashboard ==> main_ticker ::: ',main_ticker)
    print('build_dashboard ==> period ::: ',period)
    print('build_dashboard ==> compare_tickers ::: ',compare_tickers)
    
    info = get_stock_info(main_ticker)
    hist = get_stock_history(main_ticker, period)

    if info is None or hist is None:
        raise ValueError(f'{main_ticker}: 데이터를 가져올 수 없습니다.')
    
    print('build_dashboard ==> info ::: ',info)
    print('build_dashboard ==> hist ::: ',hist)

    print('2/4 메트릭 카드 생성 중...')
    # 힌트: make_metric_html(info)
    metric_html = make_metric_html(info)

    print('3/4 차트 생성 중...')
    # 통합 차트 (가격 + 거래량)
    # 힌트: make_combined_chart(hist, main_ticker, period, show_ma)
    #def make_combined_chart(df, ticker_symbol, period, show_ma=True):
    fig_main = make_combined_chart(hist,main_ticker ,period , show_ma)

    # 비교 차트
    # 힌트: make_comparison_chart(compare_tickers, period)
    #def make_comparison_chart(tickers, period='3달'):
    fig_compare = make_comparison_chart(compare_tickers , period)
    
    fig_candle = make_candle_chart(main_ticker,hist)

    # plotly 차트를 HTML div 문자열로 변환 (파일에 삽입하기 위해)
    # include_plotlyjs: 첫 번째 차트에만 plotly.js를 포함 (용량 절약)
    chart_main_html    = fig_main.to_html(
        full_html=False, include_plotlyjs='cdn' #캐싱을 하게 된다 필요할때 인터넷에서 가져와 
    )
    chart_compare_html = fig_compare.to_html(
        full_html=False, include_plotlyjs=False  # 힌트: False, False (이미 위에서 로드됨)
    )
    chart_candle_html = fig_candle.to_html(
        full_html=False,  include_plotlyjs=False
    )

    print('4/4 HTML 조립 중...')
    now = datetime.now().strftime('%Y-%m-%d %H:%M')

    # 전체 HTML 템플릿
    html = f'''
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{main_ticker} 주식 대시보드</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            background: #f0f2f5;
            margin: 0;
            padding: 24px;
        }}
        .container {{
            max-width: 1100px;
            margin: 0 auto;
        }}
        .header {{
            display: flex;
            justify-content: space-between;
            align-items: flex-end;
            margin-bottom: 20px;
        }}
        .title {{
            font-size: 26px;
            font-weight: 700;
            color: #1a1a2e;
        }}
        .subtitle {{
            font-size: 13px;
            color: #888;
        }}
        .card {{
            background: white;
            border-radius: 12px;
            padding: 20px;
            margin-bottom: 16px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.08);
        }}
    </style>
</head>
<body>
<div class="container">

    <div class="header">
        <div>
            <div class="title">{info['name']} ({main_ticker})</div>
            <div class="subtitle">조회 기간: {period} &nbsp;|&nbsp; 이동평균: {'ON' if show_ma else 'OFF'}</div>
        </div>
        <div class="subtitle">생성: {now}</div>
    </div>

    <div class="card">
        {metric_html}
    </div>

    <div class="card">
        {chart_main_html}
    </div>

    <div class="card">
        {chart_compare_html}
    </div>
    
    <div class="card">
        {chart_candle_html}
    </div>

</div>
</body>
</html>
    '''
    return html

---
## HTML 파일 저장 & 브라우저 열기

In [44]:
# 대시보드 HTML 생성
# 메인 종목 (차트 + 메트릭 카드에 표시)
# MAIN_TICKER = '005930.KS'          # 힌트: 'AAPL' 또는 '005930.KS' (삼성전자)

# # 조회 기간
# PERIOD = '3달'               # 힌트: '1주', '1달', '3달', '1년' 중 하나

# # 비교 차트에 표시할 종목 리스트
# COMPARE_TICKERS = ['AAPL', 'GOOGL', '000660.KS']   # 힌트: ['AAPL', 'TSLA', 'GOOGL']. 000660.KS -> sk하이닉스

# # 이동평균선 표시 여부
# SHOW_MA = True                # 힌트: True 또는 False

# html_content = build_dashboard(
#     main_ticker=MAIN_TICKER,            # 힌트: MAIN_TICKER
#     period=PERIOD,                      # 힌트: PERIOD
#     compare_tickers=COMPARE_TICKERS,    # 힌트: COMPARE_TICKERS
#     show_ma=SHOW_MA                     # 힌트: SHOW_MA
# )

# HTML 파일로 저장
# 힌트: open(OUTPUT_FILE, 'w', encoding='utf-8') as f: f.write(html_content)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(html_content)

import os
file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f' 저장 완료: {OUTPUT_FILE} ({file_size_kb:.1f} KB)')
print(f'   → 탐색기에서 {OUTPUT_FILE} 파일을 더블클릭해서 열어보세요!')

 저장 완료: stock_dashboard.html (40.4 KB)
   → 탐색기에서 stock_dashboard.html 파일을 더블클릭해서 열어보세요!


In [76]:
### 이번 프로젝트 심화
# 관심 종목 여러 개를 루프 돌려서 **종목별 HTML 파일 자동 생성**
# 캔들스틱 차트(`go.Candlestick`)로 교체
# `schedule` 라이브러리로 매일 오전 9시 자동 저장

#1. 관심 종목 여러 개를 루프 돌려서 **종목별 HTML 파일 자동 생성**
#= ['AAPL', 'GOOGL', '000660.KS']   # 힌트: ['AAPL', 'TSLA', 'GOOGL']. 000660.KS -> sk하이닉스
attetionStock ={
    'search_nm' : ['000660.KS','AAPL','TSLA'] 
    ,'file_nm' : ['SK hynix_stock' , 'APPL_stock','TSLA_stock']} 
for ticker , file_name in zip(attetionStock['search_nm'] ,attetionStock['file_nm']  ) :
    html_content = build_dashboard(
        main_ticker=ticker,   
        period=PERIOD,                      
        compare_tickers=COMPARE_TICKERS,    
        show_ma=SHOW_MA                     
    )

    # HTML 파일로 저장
    # 힌트: open(OUTPUT_FILE, 'w', encoding='utf-8') as f: f.write(html_content)
    with open(file_name+'.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024


#file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024


1/4 데이터 수집 중...
build_dashboard ==> main_ticker :::  000660.KS
build_dashboard ==> period :::  3달
build_dashboard ==> compare_tickers :::  ['AAPL', 'GOOGL', '000660.KS']
build_dashboard ==> info :::  {'name': 'SK hynix', 'current_price': 1819000.0, 'prev_close': 1970000.0, 'currency': 'KRW'}
build_dashboard ==> hist :::                                     Open          High           Low  \
Date                                                                  
2026-02-19 00:00:00+09:00  9.033331e+05  9.113184e+05  8.863644e+05   
2026-02-20 00:00:00+09:00  8.903570e+05  9.532410e+05  8.793773e+05   
2026-02-23 00:00:00+09:00  9.781950e+05  9.781950e+05  9.392669e+05   
2026-02-24 00:00:00+09:00  9.502466e+05  1.003149e+06  9.442576e+05   
2026-02-25 00:00:00+09:00  1.010136e+06  1.036088e+06  1.002151e+06   
2026-02-26 00:00:00+09:00  1.024000e+06  1.099000e+06  1.023000e+06   
2026-02-27 00:00:00+09:00  1.050000e+06  1.096000e+06  1.048000e+06   
2026-03-03 00:00:00+09:00  1.022000e+0

In [28]:
# (선택) 브라우저 자동 열기
import webbrowser, os

file_path = os.path.abspath(OUTPUT_FILE)
webbrowser.open(f'file://{file_path}')
print(f'브라우저로 열기: {file_path}')

0:95: execution error: 일부 대상체 파일을 발견할 수 없습니다. (-43)


브라우저로 열기: /Users/macbook/Desktop/ai_engineering_project/4주차/stock_dashboard.html


---
## 노트북 안에서 미리보기 (선택)

저장한 파일을 브라우저에서 열지 않고 노트북 안에서 바로 확인하고 싶을 때.

In [ ]:
from IPython.display import IFrame

# 저장한 HTML 파일을 노트북 안에 임베드
# 힌트: IFrame(OUTPUT_FILE, width='100%', height=1200)
IFrame(OUTPUT_FILE, width='100%', height=1200)

---
## 최종 완성 체크리스트

| 체크 | 항목 |
|:---:|------|
| ☐ | `make_metric_html()` — 카드 3개 노트북 안에 렌더링됨 |
| ☐ | `build_dashboard()` — 오류 없이 실행 완료 |
| ☐ | `stock_dashboard.html` 파일 생성됨 |
| ☐ | 브라우저에서 HTML 열면 차트 2개 + 카드 표시됨 |
| ☐ | 차트 마우스 오버 시 데이터 팝업 표시됨 |
| ☐ | 비교 차트에 여러 종목 선이 표시됨 |
| ☐ | 등락 방향에 따라 카드 색상이 달라짐 |

---

## 다음에 도전해볼 것들

### 이번 프로젝트 심화
- 관심 종목 여러 개를 루프 돌려서 **종목별 HTML 파일 자동 생성**
- 캔들스틱 차트(`go.Candlestick`)로 교체
- `schedule` 라이브러리로 매일 오전 9시 자동 저장

### 차주 Streamlit 강의 연계
> 지금 만든 `build_dashboard()`의 함수들을 그대로 재활용해서,  
> Streamlit으로 **실시간 조작이 되는 웹앱**으로 업그레이드합니다.  
> 종목을 바꾸면 차트가 즉시 갱신되는 인터랙티브 앱이 됩니다!